# KrishiSetu AI - Crop Disease Model (Colab training + export)

End-to-end notebook: it downloads the datasets, trains a MobileNetV2 classifier for
the Odisha crops, exports it to **TensorFlow.js Layers format**, and hands you a zip
you can drop straight into `KrishiSetu-AI/public/model/`.

## What you end up with

| File | Required | Notes |
|---|---|---|
| `model.json` | yes | TF.js Layers model, loaded by the app via `tf.loadLayersModel` |
| `group1-shard1of1.bin` | yes | weight shard(s). If weights exceed 4 MB you get `shardNofM` instead |
| `classes.json` | yes | plain JSON array, in the model's output order |
| `metadata.json` | optional | training info, ignored by the app |

## Preprocessing contract - do not break this

| Stage | Value |
|---|---|
| App resizes the photo to | 224 x 224 |
| App divides pixels by | 255 -> floats in [0, 1] |
| This model's FIRST layer | `Rescaling(scale=2.0, offset=-1.0)` -> [-1, 1] |
| MobileNetV2 expects | [-1, 1] |

For reference, this is the app side (`src/services/modelStorageService.js`):

```
tf.browser.fromPixels(img)
  .resizeNearestNeighbor([224, 224])
  .toFloat()
  .expandDims(0)
  .div(255.0)
```

Because the `Rescaling` layer is baked into the exported model, **the app needs no
change**. If you remove it, offline accuracy collapses.

## How to run

1. `Runtime > Change runtime type > T4 GPU > Save`.
2. Run the cells in order (`Runtime > Run all` also works).
3. Datasets: Step 2 tries the Kaggle API, then any archive or extracted folder you put in
`/content`, then a no-account fallback. See Step 2 for the details.
4. The last cell downloads `tfjs_model.zip`.

**Prove it works first:** set `SMOKE_TEST = True` in Step 0 to run the whole pipeline
with 2 + 1 epochs and 60 images per class, then rerun with `False` for the real model.


---
## Step 0 - Settings

Everything you normally need to change lives in the next cell.


In [ ]:
# ============================================================
# STEP 0 - Settings. This is the only cell you normally edit.
# ============================================================
EPOCHS_P1 = 10            # phase 1: train the new head, MobileNetV2 frozen
EPOCHS_P2 = 5             # phase 2: fine-tune the top of MobileNetV2
BATCH_SIZE = 32
IMG_SIZE = (224, 224)     # must stay 224 - the app resizes photos to 224x224
VAL_SPLIT = 0.2
SEED = 42
CAP_PER_CLASS = 1500      # max images per class (keeps Colab time + RAM sane)
ALPHA = 1.0               # MobileNetV2 width: 1.0 = 2.26M params (~4.5MB at
                          # float16), 0.5 = ~0.7M params (~1.5MB). The alpha=0.5
                          # model scored 82.7% - under the 85% threshold MODEL_PLAN.md
                          # section 4 sets - so 1.0 is now the default. It is not
                          # free: the app precaches /model/*.bin in its service worker,
                          # so 1.0 costs roughly +3MB on first install. Put 0.5 back
                          # if download size matters more than accuracy.
EXPORT_FLOAT16 = True     # halves the weight file size; the app loads it fine
BACKUP_TO_DRIVE = False   # True = also copy the zip to MyDrive (survives a disconnect)
SMOKE_TEST = False        # True = 2 + 1 epochs and 60 images/class, to test the pipeline
UPLOAD_NOW = False        # True = show a file picker in Step 2 so you can upload zips by hand
USE_TFDS_FALLBACK = True  # also pull PlantVillage for any picker crop still missing (maize)
DROP_CLASSES_WITHOUT_REMEDY = True  # skip classes the app has no remedy for

# ---------------------------------------------------------------------------
# Field realism + the Other class (see MODEL_PLAN.md sections 2, 3 and 4)
# ---------------------------------------------------------------------------
# PlantVillage is studio photos of single leaves on white paper. A real paddy
# photo is dim, tilted and cluttered. These layers close part of that gap; they
# are a stopgap, not a substitute for photos taken with your own phone.
FIELD_AUGMENT = True      # False = the old, gentler augmentation
AUG_BRIGHTNESS = 0.25     # +-25% exposure - field light swings far more than a studio
AUG_CONTRAST = 0.25
AUG_ROTATION = 0.15       # +-~27 degrees; people hold phones at an angle
AUG_TRANSLATION = 0.10    # off-centre framing
AUG_ZOOM = 0.20
AUG_NOISE = 8.0           # sensor noise stddev, in 0-255 units - see the sensor_noise layer

# One extra class that means "this is not a leaf of the crop you picked".
# The app checks it on RAW probabilities before the crop mask
# (GLOBAL_PREFIXES / OTHER_MIN_CONFIDENCE in src/services/offlineDiagnosis.js), so
# the name must start with Other_ .
TRAIN_OTHER_CLASS = True
OTHER_CLASS = 'Other_NotALeaf'
# Upload 150-300 real negatives here (soil, sky, a hand, other plants, blurry
# shots). Anything found is used; anything missing is replaced by synthetic
# placeholders and the notebook says so in its output.
NEGATIVE_DIRS = ['/content/raw/negatives', '/content/raw/other', '/content/other_negatives']
NEGATIVE_LIMIT = 800

# Optional: point this at the app's src/data/offline_diseases.json (upload it, or
# clone the repo in Colab) to have Step 3b print which records could be merged
# because they carry the same treatment. Today the answer is "none" - see
# MODEL_PLAN.md section 2.
DICTIONARY_PATH = ''

# ---------------------------------------------------------------------------
# KAGGLE CREDENTIALS - this is the spot for your KGAT_ token
# ---------------------------------------------------------------------------
# Get one at: kaggle.com > click your avatar > Settings > API > Create New Token
# It starts with KGAT_. Paste it between the quotes below - that is the whole
# setup. Step 2 exports it as KAGGLE_API_TOKEN for the kaggle CLI (and stashes it
# in ~/.kaggle/access_token), then downloads the datasets.
#
# An access token is a Bearer credential - it is NOT a kaggle.json
# username + key pair, so pasting it into a kaggle.json will never work.
#
# SECURITY: this file is tracked by git. Before you publish the repo, set
# KAGGLE_API_TOKEN back to '' and rotate the token on Kaggle. Step 2 also reads
# the Colab secret KAGGLE_API_TOKEN, an env var, or /content/kaggle_token.txt,
# so you can move it out of this file without changing any code.
KAGGLE_API_TOKEN = ''

# Old-style kaggle.json credentials (username + key). Only needed if you have
# those instead of a KGAT_ token - leave both '' otherwise.
KAGGLE_USERNAME = ''
KAGGLE_KEY = ''

# Kaggle datasets as owner/slug. Set a value to None to skip that dataset.
#
# abdallahalidev/plantvillage-dataset is the FULL 38-class PlantVillage (54,303
# images) and is the only Kaggle source here with corn/maize. The old
# emmarex/plantdisease is a 15-class subset - Pepper, Potato, Tomato, no corn -
# shipped as two nested copies of every folder.
#
# It is spMohanty/PlantVillage-Dataset zipped, so it has color/, grayscale/ and
# segmented/ copies of every class. Step 3 keeps color/ only: the copies share
# a filename but differ in file size, so the duplicate filter cannot merge them
# and the same leaf would otherwise land on both sides of our split. If the
# download 403s, open the dataset page once in a browser and accept its rules.
#
# Do NOT swap in vipoooool/new-plant-diseases-dataset: its own description says
# it is offline-augmented and pre-split 80/20, which straddles our split with
# near-copies of the same leaf.
KAGGLE_DATASETS = {
    'plantvillage': 'abdallahalidev/plantvillage-dataset',
    'rice': 'anshulm257/rice-disease-dataset',
    'cotton': 'seroshkarim/cotton-leaf-disease-dataset',
}

# File extensions we treat as images
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

if SMOKE_TEST:
    EPOCHS_P1, EPOCHS_P2, CAP_PER_CLASS = 2, 1, 60
    print('SMOKE TEST MODE: 2 + 1 epochs, 60 images per class')

print('Settings: ' + str(EPOCHS_P1) + ' + ' + str(EPOCHS_P2) + ' epochs, cap '
      + str(CAP_PER_CLASS) + ' images/class, alpha ' + str(ALPHA))
print('Other class: ' + (OTHER_CLASS if TRAIN_OTHER_CLASS else 'off')
      + '  |  field augmentation: ' + ('on' if FIELD_AUGMENT else 'off'))


---
## Step 1 - Install the TensorFlow.js converter

The converter (`tensorflowjs`) is what writes `model.json` + the `.bin` shards.
It needs a few workarounds on a current Colab runtime; they are all handled below.
Run this cell, and if it tells you to restart the runtime, do that and run it again.


In [ ]:
# ============================================================
# STEP 1 - Install tensorflowjs and check the environment
# ============================================================
# Why this is not a plain pip install:
#   1. tensorflowjs 4.x declares python_requires < 3.13, and Colab is on 3.13+, so
#      pip needs --ignore-requires-python.
#   2. Its dependency pins conflict with Colab's stack (ResolutionImpossible),
#      so install with --no-deps and supply the two real dependencies by hand.
#   3. The converter hard-imports tensorflow_decision_forests, which ships no
#      3.13 wheel. We convert a MobileNetV2 (no decision forests) -> stub it.
#   4. numpy 2.x removed the np.object / np.bool aliases that old tf-hub uses.
#   5. pip does not refresh modules already imported in this kernel session, so
#      purge the stale entries before importing.
import importlib.metadata as _im
import subprocess
import sys


def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args],
                       capture_output=True, text=True)
    tail = ((r.stdout or '') + (r.stderr or ''))[-1200:]
    if tail.strip():
        print(tail)
    return r


pip('uninstall', '-y', 'tensorflowjs')
r = pip('install', '--ignore-requires-python', '--no-deps', 'tensorflowjs==4.22.0')
assert r.returncode == 0, 'tensorflowjs install failed - read the output above'
pip('install', 'tensorflow-hub>=0.16.1', 'six', 'importlib_resources')

# what pip actually put on disk, before we import it
print('tensorflowjs on disk: ' + _im.version('tensorflowjs'))

# purge modules pip just replaced (they stay cached in this kernel otherwise)
for _m in [m for m in list(sys.modules)
           if m.split('.')[0] in ('tensorflowjs', 'tensorflow_hub')]:
    del sys.modules[_m]

import types
import warnings
import numpy as np
import tensorflow as tf

with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    for _name, _type in [('object', object), ('bool', bool), ('int', int),
                         ('float', float), ('str', str)]:
        try:
            getattr(np, _name)
        except AttributeError:
            setattr(np, _name, _type)

if not hasattr(tf.compat.v1, 'estimator'):
    _estimator_stub = types.ModuleType('estimator_stub')

    class _Exporter:
        pass

    _estimator_stub.Exporter = _Exporter
    tf.compat.v1.estimator = _estimator_stub


def stub_module(name):
    mod = types.ModuleType(name)
    mod.__getattr__ = lambda attr: stub_module(name + '.' + attr)
    return mod


sys.modules.setdefault('tensorflow_decision_forests',
                       stub_module('tensorflow_decision_forests'))

import glob
import json
import os
import random
import shutil
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflowjs as tfjs
from sklearn.metrics import classification_report, confusion_matrix

print('Python       : ' + sys.version.split()[0])
print('TensorFlow   : ' + tf.__version__)
print('tensorflowjs : ' + tfjs.__version__ + '  (from ' + str(tfjs.__file__) + ')')
print('NumPy        : ' + np.__version__)
print('GPU devices  : ' + str(tf.config.list_physical_devices('GPU')))

assert tfjs.__version__.startswith('4.'), (
    'Imported a stale tensorflowjs ' + tfjs.__version__ + '. Do Runtime > Restart '
    'session, then re-run this cell from the top.')

if not tf.config.list_physical_devices('GPU'):
    print()
    print('WARNING: no GPU detected. Training will be extremely slow.')
    print('Runtime > Change runtime type > T4 GPU > Save.')
else:
    print()
    print('Environment ready.')


---
## Step 2 - Get the datasets

Four ways in, all handled by the next cell, in this order:

1. **Kaggle API** - automatic, gets all three datasets. **Paste your `KGAT_` access token
into `KAGGLE_API_TOKEN` in Step 0** - that is the whole setup. You can also upload a
`kaggle.json` to
`/content`, or set the Colab secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (key icon in the
left sidebar, enable notebook access). Token comes from
`kaggle.com > your profile > Settings > API > Create New Token`.

An access token is a `Bearer` credential, **not** a username/key pair, so it must never be
put into a `kaggle.json`. Step 2 writes it to `~/.kaggle/access_token`, exports
`KAGGLE_API_TOKEN` for the kaggle CLI, and if the CLI refuses it retries the download
against the API directly with the token in an `Authorization: Bearer` header. It also
accepts the Colab secret `KAGGLE_API_TOKEN`, an environment variable, or a raw token in
`/content/kaggle_token.txt`, so the token can live outside this notebook.
2. **Archives you uploaded** - any `.zip` / `.tar.gz` anywhere in `/content` is detected by
file name and extracted automatically.
3. **Folders you uploaded** - a dataset folder you already extracted into `/content` is
used as-is.
4. **`tensorflow_datasets`** - the full 38-class PlantVillage with no account at all:
**828 MiB download**, a few minutes, needs `USE_TFDS_FALLBACK = True` (the default). Of
the app's five crops it covers maize/tomato/potato, so Paddy and Cotton still come from
Kaggle. It runs whenever a picker crop has no folder on disk, not only when nothing else
loaded, so the maize classes arrive even after a successful Kaggle download.

| Dataset | Kaggle slug | Images | Classes | Crops we use |
|---|---|---|---|---|
| PlantVillage (subset) | `emmarex/plantdisease` | 41,276 raw -> 18,947 after de-duplication | 15 (+15 nested copies) | Tomato, Potato |
| Rice leaf diseases | `anshulm257/rice-disease-dataset` | 3,829 | 6 | Paddy |
| Cotton leaf disease | `seroshkarim/cotton-leaf-disease-dataset` | 1,710 | 4 | Cotton |
| PlantVillage (full, no account) | *via `tensorflow_datasets`* | 54,303 | 38 | Maize, Tomato, Potato |

Notes on that table:

* **`emmarex/plantdisease` contains no corn/maize at all.** It is a 15-class subset -
  Pepper (2), Potato (3), Tomato (10) - shipped as two nested copies of every folder, which
  is where the `skipped 18163 duplicate file(s)` line comes from. Pepper is skipped
  because the app has no pepper crop. This is the single reason a model trained only from
  the three Kaggle rows has zero `Maize_*` classes.
* **Rice** produces 6 folders; `Leaf scald` is dropped at training time because
  `src/data/offline_diseases.json` has no record for it (`DROP_CLASSES_WITHOUT_REMEDY`).
* **Cotton** produces 4 folders. One is misspelled `fussarium_wilt` upstream;
  `classify()` accepts both `fusarium` and `fussarium`, so it is no longer skipped.
* **Row 4** is what fills the maize gap. Step 2 prints crop coverage twice - before and
  after - so `ok Maize` appears before you move on, and Step 9 repeats the check at the
  end.

### The two 38-class Kaggle mirrors, and why they are not used

Both really do have 38 classes, and both corrupt a random train/val split:

* `abdallahalidev/plantvillage-dataset` (2.18 GB) - the
  `spMohanty/PlantVillage-Dataset` repo zipped, so it ships `color/`, `grayscale/` and
  `segmented/` copies of every class. This notebook walks all three and the variants
  differ in file size, so the duplicate filter cannot merge them: every leaf gets trained
  three times, twice as grayscale or background-removed.
* `vipoooool/new-plant-diseases-dataset` (1.43 GB) - its own description says it was
  "recreated using offline augmentation" and is pre-split 80/20, so near-copies of one leaf
  land on both sides of our split.

To use either anyway: add a path filter that skips `/grayscale/` and `/segmented/`, and
**remove `emmarex/plantdisease` from `KAGGLE_DATASETS`** so the two copies of
PlantVillage do not merge into one pool.

If nothing at all is found, the cell prints those options and stops with a clear error
instantly instead of failing later during training. If a Kaggle download returns 403, open
that dataset page in a browser once and accept its rules.


In [ ]:
# ============================================================
# STEP 2 - Download / extract the datasets into /content/raw
# ============================================================
RAW = '/content/raw'
os.makedirs(RAW, exist_ok=True)


def count_images(root):
    return sum(len([f for f in files if f.lower().endswith(IMG_EXTS)])
               for _, _, files in os.walk(root))


def kaggle_rest_download(slug, dest):
    # Fallback for when the kaggle CLI fails: fetch the dataset zip straight from
    # the API with the credential in an Authorization header.
    import base64
    import urllib.request
    req = urllib.request.Request(
        'https://www.kaggle.com/api/v1/datasets/download/' + slug)
    if KAGGLE_TOKEN:
        req.add_header('Authorization', 'Bearer ' + KAGGLE_TOKEN)
    elif LEGACY:
        pair = (LEGACY['username'] + ':' + LEGACY['key']).encode()
        req.add_header('Authorization', 'Basic ' + base64.b64encode(pair).decode())
    tmp = '/content/_kaggle_download.zip'
    try:
        with urllib.request.urlopen(req, timeout=900) as resp, open(tmp, 'wb') as f:
            shutil.copyfileobj(resp, f)
        with zipfile.ZipFile(tmp) as zf:
            zf.extractall(dest)
        return count_images(dest) > 0
    except Exception as exc:
        print('  REST fallback failed: ' + str(exc))
        return False
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


# ---- A) Kaggle credentials ------------------------------------------------
# Resolved in this order:
#   1. KAGGLE_API_TOKEN from Step 0      (new style, starts with KGAT_)
#   2. KAGGLE_API_TOKEN from an env var or a Colab secret
#   3. KAGGLE_API_TOKEN from /content/kaggle_token.txt or ~/.kaggle/access_token
#   4. old style: KAGGLE_USERNAME + KAGGLE_KEY (Step 0, env, Colab secret)
#   5. old style: a kaggle.json uploaded to /content or ~/.kaggle
KAGGLE_TOKEN = None
LEGACY = None


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


for _candidate in [KAGGLE_API_TOKEN, os.environ.get('KAGGLE_API_TOKEN'),
                   colab_secret('KAGGLE_API_TOKEN')]:
    if _candidate and str(_candidate).strip():
        KAGGLE_TOKEN = str(_candidate).strip()
        break

if KAGGLE_TOKEN is None:
    for _path in ['/content/kaggle_token.txt', '/content/access_token',
                  os.path.expanduser('~/.kaggle/access_token')]:
        if os.path.exists(_path):
            _text = open(_path).read().strip()
            if _text:
                KAGGLE_TOKEN = _text
                print('Read the access token from ' + _path)
                break

if KAGGLE_TOKEN is not None and not KAGGLE_TOKEN.startswith('KGAT_'):
    print('WARNING: that credential does not start with KGAT_ - a Kaggle access')
    print('token always does. If it is an old-style API key, put it in KAGGLE_KEY.')
    KAGGLE_TOKEN = None

if KAGGLE_TOKEN is None:
    _user = (KAGGLE_USERNAME or os.environ.get('KAGGLE_USERNAME')
             or colab_secret('KAGGLE_USERNAME'))
    _key = (KAGGLE_KEY or os.environ.get('KAGGLE_KEY')
            or colab_secret('KAGGLE_KEY'))
    if _user and _key:
        LEGACY = {'username': _user, 'key': _key}
    else:
        for _path in ['/content/kaggle.json',
                      os.path.expanduser('~/.kaggle/kaggle.json')]:
            if os.path.exists(_path):
                LEGACY = json.load(open(_path))
                print('Found old-style credentials at ' + _path)
                break

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
if KAGGLE_TOKEN:
    with open(os.path.join(kaggle_dir, 'access_token'), 'w') as f:
        f.write(KAGGLE_TOKEN)
    os.chmod(os.path.join(kaggle_dir, 'access_token'), 0o600)
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
    print('Kaggle auth: access token ' + KAGGLE_TOKEN[:9] + '...'
          + KAGGLE_TOKEN[-4:])
elif LEGACY:
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump(LEGACY, f)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print('Kaggle auth: username + key (' + str(LEGACY.get('username')) + ')')
else:
    print('No Kaggle credentials found - checking for uploads instead.')
    print('For Paddy and Cotton too, paste your KGAT_ token into KAGGLE_API_TOKEN')
    print('in Step 0 and re-run this cell.')

if KAGGLE_TOKEN or LEGACY:
    pip('install', '-q', '-U', 'kaggle')

    for name, slug in KAGGLE_DATASETS.items():
        if not slug:
            continue
        dest = os.path.join(RAW, name)
        if os.path.isdir(dest) and count_images(dest):
            print(name + ': already present (' + str(count_images(dest))
                  + ' images), skipping')
            continue
        os.makedirs(dest, exist_ok=True)
        print('downloading ' + slug + ' -> ' + dest)
        r = subprocess.run(
            [sys.executable, '-m', 'kaggle', 'datasets', 'download',
             '-d', slug, '-p', dest, '--unzip'],
            capture_output=True, text=True)
        if r.returncode == 0:
            print('  ' + name + ': ' + str(count_images(dest)) + ' images')
            continue
        print('  kaggle CLI failed, trying the REST API directly...')
        if kaggle_rest_download(slug, dest):
            print('  ' + name + ': ' + str(count_images(dest)) + ' images')
            continue
        print('  download failed for ' + slug)
        print('  check the slug, and accept that dataset rules once in a browser')

# ---- B) optional: pick the zips right here in Colab
if UPLOAD_NOW:
    try:
        from google.colab import files as colab_files
        print('Pick your dataset zip(s) in the dialog...')
        for uploaded_name in (colab_files.upload() or {}):
            print('  uploaded: ' + uploaded_name)
    except Exception as exc:
        print('Upload unavailable (' + str(exc) + ')')


# ---- C) archives anywhere in /content
import tarfile


archives = sorted(glob.glob('/content/*.zip') + glob.glob('/content/*/*.zip')
                  + glob.glob('/content/*.tar.gz') + glob.glob('/content/*.tgz'))
for zpath in archives:
    base = os.path.basename(zpath).lower()
    if 'tfjs' in base or 'model' in base or 'kaggle' in base:
        continue
    if 'rice' in base or 'paddy' in base:
        target = os.path.join(RAW, 'rice')
    elif 'cotton' in base:
        target = os.path.join(RAW, 'cotton')
    else:
        target = os.path.join(RAW, 'plantvillage')
    if os.path.exists(os.path.join(RAW, '.extracted_' + base.replace('.', '_'))):
        print('skipping ' + base + ' - already extracted')
        continue
    os.makedirs(target, exist_ok=True)
    print('extracting ' + base + ' -> ' + target)
    if base.endswith('.zip'):
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(target)
    else:
        with tarfile.open(zpath) as tf:
            tf.extractall(target)
    open(os.path.join(RAW, '.extracted_' + base.replace('.', '_')), 'w').close()


def image_folders(root):
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        if any(f.lower().endswith(IMG_EXTS) for f in filenames):
            found.append(dirpath)
    return found


# ---- D) datasets dropped straight into /content
IGNORED_DIRS = ('raw', 'odisha_crops', 'tfjs_model', 'sample_data', 'drive', '__pycache__')
loose = [os.path.join('/content', d) for d in sorted(os.listdir('/content'))
         if os.path.isdir(os.path.join('/content', d))
         and d not in IGNORED_DIRS and not d.startswith('.')]

found_images = count_images(RAW) + sum(count_images(root) for root in loose)

# ---- crop coverage, before deciding whether PlantVillage is needed --------
# The app's crop picker offers five crops and the model has to answer for all of
# them. Step 2 cannot call Step 3's classify() yet, so this is a plain keyword
# scan over folder names; Step 3 does the real top-level -> class mapping.
PICKER_CROPS = {
    'Paddy': ('paddy', 'rice'),
    'Maize': ('corn', 'maize'),
    'Tomato': ('tomato',),
    'Potato': ('potato',),
    'Cotton': ('cotton',),
}


def crop_of_text(text):
    low = str(text).lower().replace('_', ' ').replace('-', ' ')
    for crop, words in PICKER_CROPS.items():
        if any(w in low for w in words):
            return crop
    return None


def crops_present(root):
    found = set()
    if not os.path.isdir(root):
        return found
    for _, dirnames, _ in os.walk(root):
        for d in dirnames:
            crop = crop_of_text(d)
            if crop:
                found.add(crop)
    return found


def picker_coverage():
    found = set()
    for root in [RAW] + loose:
        found |= crops_present(root)
    return found


covered_crops = picker_coverage()
missing_crops = [c for c in PICKER_CROPS if c not in covered_crops]
print()
print('Crop coverage from what is already on disk:')
for _crop in PICKER_CROPS:
    print('  ' + ('ok   ' if _crop in covered_crops else 'MISS ') + _crop)

# ---- E) PlantVillage through tensorflow_datasets (no Kaggle account needed)
# This is the FULL 38-class PlantVillage (54,303 images), so it is the only
# no-account source of the maize/corn folders. Kaggle's 15-class PlantVillage
# mirror (emmarex/plantdisease) simply does not contain them - which is why a
# model trained only from Kaggle has zero Maize_* classes.
#
# It runs whenever a picker crop still has no folder, NOT only when nothing else
# loaded: with the Kaggle downloads in place found_images is never 0, so the
# maize classes would otherwise never arrive.
#
# Kaggle's 38-class mirrors are not a substitute.
# abdallahalidev/plantvillage-dataset is the spMohanty repo zipped, and that
# ships color/ + grayscale/ + segmented/ copies of every class. This notebook
# walks all three, and because the variants differ in file size the duplicate
# filter cannot merge them - so every leaf gets trained three times, twice as
# grayscale and background-removed.
# vipoooool/new-plant-diseases-dataset is offline-augmented and pre-split 80/20.
# Both put near-copies of one leaf on both sides of our split.
TFDS_DEST = os.path.join(RAW, 'plantvillage_tfds')
TFDS_CROPS = ('Maize', 'Tomato', 'Potato')  # the picker crops PlantVillage has
tfds_wanted = [c for c in missing_crops if c in TFDS_CROPS]
tfds_missing = [c for c in tfds_wanted if c not in crops_present(TFDS_DEST)]

if USE_TFDS_FALLBACK and tfds_wanted and tfds_missing:
    print()
    print('Fetching PlantVillage through tensorflow_datasets for: '
          + ', '.join(tfds_missing))
    print('(no Kaggle account needed, ~830 MB, a few minutes).')
    try:
        import tensorflow_datasets as tfds
        from PIL import Image as PILImage

        dest = TFDS_DEST
        os.makedirs(dest, exist_ok=True)
        left_out = set()
        builder = tfds.builder('plant_village')
        builder.download_and_prepare()
        label_names = builder.info.features['label'].names
        written = 0
        for split in builder.info.splits:
            data = builder.as_dataset(split=split, as_supervised=True)
            for i, (image, label) in enumerate(tfds.as_numpy(data)):
                name = label_names[int(label)]
                # Keep only the crops that are still uncovered. Bringing the
                # tomato and potato folders in as well would double them: this
                # is the same PlantVillage the Kaggle mirror is built from but
                # with different filenames, so the duplicate filter would not
                # catch them and one leaf would land on both sides of the split.
                if crop_of_text(name) not in tfds_missing:
                    left_out.add(crop_of_text(name) or name.split('___')[0])
                    continue
                out_dir = os.path.join(dest, name)
                os.makedirs(out_dir, exist_ok=True)
                PILImage.fromarray(image).save(
                    os.path.join(out_dir,
                                 name + '_' + split + '_' + str(i).zfill(5) + '.jpg'),
                    'JPEG', quality=92)
                written += 1
        print('plant_village -> ' + str(written) + ' images in ' + dest)
        print('  kept    : ' + ', '.join(tfds_missing))
        print('  left out: ' + ', '.join(sorted(left_out)))
        print('NOTE: PlantVillage can never cover Paddy or Cotton - those still need')
        print('the rice and cotton datasets from Kaggle.')
        found_images = count_images(RAW)
    except Exception as exc:
        print('tensorflow_datasets fallback failed: ' + str(exc))

# ---- inventory -----------------------------------------------------------
print()
print('Dataset inventory:')
DATA_ROOTS = []
for entry in sorted(os.listdir(RAW)):
    full = os.path.join(RAW, entry)
    if not os.path.isdir(full):
        continue
    n_images = count_images(full)
    if n_images:
        DATA_ROOTS.append(full)
        print('  ' + entry.ljust(18) + str(len(image_folders(full))).rjust(5)
              + ' folders, ' + str(n_images).rjust(7) + ' images')

for root in loose:
    n_images = count_images(root)
    if n_images:
        DATA_ROOTS.append(root)
        print('  ' + os.path.basename(root).ljust(18)
              + str(len(image_folders(root))).rjust(5)
              + ' folders, ' + str(n_images).rjust(7) + ' images   (from /content)')

if not DATA_ROOTS:
    print()
    print('=' * 68)
    print('NO DATASET FOUND - pick one option and re-run this cell')
    print('=' * 68)
    print('A) Kaggle API (best: includes Paddy and Cotton)')
    print('   kaggle.com > your profile > Settings > API > Create New Token,')
    print('   then upload the downloaded kaggle.json to /content and re-run.')
    print('   (or set the Colab secrets KAGGLE_USERNAME and KAGGLE_KEY)')
    print('B) Upload by hand')
    print('   set UPLOAD_NOW = True in Step 0, re-run this cell, pick your zips.')
    print('   kaggle.com/datasets/emmarex/plantdisease')
    print('   kaggle.com/datasets/anshulm257/rice-disease-dataset')
    print('   kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset')
    print('C) No account at all')
    print('   leave USE_TFDS_FALLBACK = True in Step 0: PlantVillage is downloaded')
    print('   through tensorflow_datasets. Maize / Tomato / Potato only.')
    print('=' * 68)
    raise RuntimeError('No dataset found - see the three options printed above.')

# ---- re-check coverage now that anything extra has landed on disk --------
covered_crops = picker_coverage()
missing_crops = [c for c in PICKER_CROPS if c not in covered_crops]
print()
print('Crop coverage for the picker (' + str(len(PICKER_CROPS)) + ' crops):')
for _crop in PICKER_CROPS:
    print('  ' + ('ok   ' if _crop in covered_crops else 'MISS ') + _crop)
if missing_crops:
    print('  -> still missing: ' + ', '.join(missing_crops))
    print('     Step 9 warns about these. Paddy and Cotton need their own Kaggle')
    print('     datasets; tomato / potato / maize come from PlantVillage.')
else:
    print('  every crop in the picker has folders to train on.')

print()
print('Data roots for Step 3:')
for root in DATA_ROOTS:
    print('  ' + root)
print('Step 2 done.')


---
## Step 3 - Build the training set

Every source folder is mapped to a `Crop_Disease` class name and its images are copied
into `/content/odisha_crops/<Crop_Disease>/`. Folder names are the labels - `classes.json`
is generated from them later - so they must match the crops used by the app's remedy
dictionary (`src/data/offline_diseases.json`): `Paddy_`, `Maize_`, `Tomato_`, `Potato_`,
`Cotton_`.

Folders that map to no known crop+disease are skipped on purpose (apple, grape, pepper,
aphids, powdery mildew ...). They are counted and listed so you can add a rule if you
want to keep one.

The same applies to a class the app has no remedy for. With
`DROP_CLASSES_WITHOUT_REMEDY = True` (the default) Step 3 drops it, because predicting a
disease and then showing the farmer the *wrong* treatment is worse than not predicting it
at all. The dropped classes are listed with the record that almost matched.

`abdallahalidev/plantvillage-dataset` (the full 38-class PlantVillage) ships `color/`,
`grayscale/` and `segmented/` copies of every class. Only `color/` is used: the three
copies share a filename but differ in file size, so the duplicate filter cannot merge
them, and training on grayscale or background-removed copies of a leaf already in the
pool would put near-identical images on both sides of our split.

Whatever structure the sources arrive in, everything is pooled here and re-split by this
notebook. It never trusts a train/valid folder somebody else created, which is what keeps
the 80/20 split honest.


In [ ]:
# ============================================================
# STEP 3 - Build /content/odisha_crops/<Crop_Disease>/
# ============================================================
odisha_dir = '/content/odisha_crops'
shutil.rmtree(odisha_dir, ignore_errors=True)
os.makedirs(odisha_dir, exist_ok=True)

# ---- the app's remedy dictionary, embedded --------------------------------
# Step 9 matches every trained class against src/data/offline_diseases.json so
# the app can turn a prediction into treatment advice. Only crop_name and
# disease_name take part in that matching, so both are embedded here and there
# is nothing to upload. If you drop the real dictionary at
# /content/offline_diseases.json, Step 9 uses that instead - the only difference.
import re

REMEDY_DB = [
    {'crop_name': 'Paddy', 'disease_name': 'Bacterial Leaf Blight (Xanthomonas oryzae pv. oryzae)'},
    {'crop_name': 'Paddy', 'disease_name': 'Leaf Blight (Helminthosporium oryzae)'},
    {'crop_name': 'Paddy', 'disease_name': 'Brown Spot (Bipolaris oryzae)'},
    {'crop_name': 'Paddy', 'disease_name': 'Rice Blast (Magnaporthe oryzae)'},
    {'crop_name': 'Paddy', 'disease_name': 'Leaf Smut (Entyloma oryzae)'},
    {'crop_name': 'Paddy', 'disease_name': 'Tungro Virus'},
    {'crop_name': 'Paddy', 'disease_name': 'Hispa (Dicladispa armigera)'},
    {'crop_name': 'Paddy', 'disease_name': 'Sheath Blight (Rhizoctonia solani)'},
    {'crop_name': 'Paddy', 'disease_name': 'Leaf Scald (Rhizoctonia oryzae-sativae)'},
    {'crop_name': 'Paddy', 'disease_name': 'Healthy'},
    {'crop_name': 'Maize', 'disease_name': 'Common Rust (Puccinia sorghi)'},
    {'crop_name': 'Maize', 'disease_name': 'Northern Leaf Blight (Exserohilum turcicum)'},
    {'crop_name': 'Maize', 'disease_name': 'Gray Leaf Spot (Cercospora zeae-maydis)'},
    {'crop_name': 'Maize', 'disease_name': 'Healthy'},
    {'crop_name': 'Tomato', 'disease_name': 'Bacterial Spot (Xanthomonas campestris pv. vesicatoria)'},
    {'crop_name': 'Tomato', 'disease_name': 'Early Blight (Alternaria solani)'},
    {'crop_name': 'Tomato', 'disease_name': 'Late Blight (Phytophthora infestans)'},
    {'crop_name': 'Tomato', 'disease_name': 'Leaf Mold (Passalora fulva)'},
    {'crop_name': 'Tomato', 'disease_name': 'Septoria Leaf Spot (Septoria lycopersici)'},
    {'crop_name': 'Tomato', 'disease_name': 'Spider Mite (Tetranychus urticae)'},
    {'crop_name': 'Tomato', 'disease_name': 'Target Spot (Corynespora cassiicola)'},
    {'crop_name': 'Tomato', 'disease_name': 'Yellow Leaf Curl Virus (TYLCV)'},
    {'crop_name': 'Tomato', 'disease_name': 'Mosaic Virus (ToMV)'},
    {'crop_name': 'Tomato', 'disease_name': 'Healthy'},
    {'crop_name': 'Potato', 'disease_name': 'Early Blight (Alternaria solani)'},
    {'crop_name': 'Potato', 'disease_name': 'Late Blight (Phytophthora infestans)'},
    {'crop_name': 'Potato', 'disease_name': 'Healthy'},
    {'crop_name': 'Cotton', 'disease_name': 'Cotton Leaf Curl Virus (CLCuV)'},
    {'crop_name': 'Cotton', 'disease_name': 'Bacterial Blight (Xanthomonas axonopodis pv. malvacearum)'},
    {'crop_name': 'Cotton', 'disease_name': 'Fusarium Wilt (Fusarium oxysporum f.sp. vasinfectum)'},
    {'crop_name': 'Cotton', 'disease_name': 'Healthy'},
]


def app_normalize(s):
    # Mirrors the normalisation in src/services/offlineDiagnosis.js
    s = str(s or '').lower()
    s = re.sub('[^a-z0-9 ]', ' ', s)
    return ' ' + ' '.join(s.split()) + ' '


def app_lookup(predicted):
    # Mirrors lookupProtocol() in src/services/offlineDiagnosis.js
    pred = app_normalize(predicted)
    words = pred.split()
    crop_word = words[0] if words else ''
    best, best_score = None, 0
    for rec in REMEDY_DB:
        crop = app_normalize(rec.get('crop_name')).split()
        if crop and crop[0] != crop_word:
            continue
        hay = app_normalize(rec.get('disease_name')).split()
        score = 0
        for i, w in enumerate(hay):
            nxt = hay[i + 1] if i + 1 < len(hay) else ''
            bigram = (w + ' ' + nxt).strip()
            # The crop word is not evidence of a disease match. The Cotton
            # record's disease_name starts with the crop name, so counting it
            # would let Cotton_Healthy score the Leaf Curl record on the crop
            # prefix alone - and that record comes first, so the real Healthy
            # record (which ties on score) could never win. A wrong pesticide
            # dose is the worst thing this app can hand out.
            if w != crop_word and len(w) > 2 and w in words:
                score += 1
            if ' ' in bigram and w != crop_word and nxt != crop_word and (' ' + bigram + ' ') in pred:
                score += 2
        if score > best_score:
            best, best_score = rec, score
    return best, best_score


GENERIC_WORDS = {'leaf', 'leaves', 'plant', 'plants', 'healthy', 'disease'}


def remedy_quality(predicted, rec):
    # The app scores the crop word twice - once as the crop, once inside the
    # disease name - so a bare crop match is not enough: Cotton_Healthy would
    # 'match' Cotton Leaf Curl Virus. Require a shared word that actually names
    # the disease, or a Healthy-to-Healthy match.
    crop_word = app_normalize(predicted).split()[0]

    def distinctive(text):
        return set(w for w in app_normalize(text).split()
                   if w != crop_word and len(w) > 2 and w not in GENERIC_WORDS)

    hay, mine = distinctive(rec.get('disease_name')), distinctive(predicted)
    return bool(hay & mine) or not (hay | mine)


def norm(text):
    text = str(text).lower().replace('_', ' ').replace('-', ' ')
    for ch in '()[],':
        text = text.replace(ch, ' ')
    return ' '.join(text.split())


def has_all(text, words):
    return all(w in text for w in words)


def classify(text):
    # 1) crop - the first token of the class name, which is what the app matches on
    if has_all(text, ['rice']) or has_all(text, ['paddy']):
        crop = 'Paddy'
    elif has_all(text, ['corn']) or has_all(text, ['maize']):
        crop = 'Maize'
    elif has_all(text, ['tomato']):
        crop = 'Tomato'
    elif has_all(text, ['potato']):
        crop = 'Potato'
    elif has_all(text, ['cotton']):
        crop = 'Cotton'
    else:
        return None

    # 2) disease - specific rules first, generic last
    if has_all(text, ['bacterial', 'blight']):
        dis = 'Bacterial_Blight'
    elif has_all(text, ['bacterial', 'spot']):
        dis = 'Bacterial_Spot'
    elif has_all(text, ['brown', 'spot']):
        dis = 'Brown_Spot'
    elif has_all(text, ['blast']):
        dis = 'Blast'
    elif has_all(text, ['smut']):
        dis = 'Leaf_Smut'
    elif has_all(text, ['tungro']):
        dis = 'Tungro'
    elif has_all(text, ['hispa']):
        dis = 'Hispa'
    elif has_all(text, ['sheath']):
        dis = 'Sheath_Blight'
    elif has_all(text, ['scald']):  # rice leaf scald - the app has no remedy record for it yet
        dis = 'Leaf_Scald'
    elif has_all(text, ['northern']):
        dis = 'Leaf_Blight'
    elif has_all(text, ['gray', 'leaf']) or has_all(text, ['grey', 'leaf']):
        dis = 'Gray_Leaf_Spot'
    elif has_all(text, ['cercospora']):
        dis = 'Gray_Leaf_Spot'
    elif has_all(text, ['rust']):
        dis = 'Rust'
    elif has_all(text, ['early', 'blight']):
        dis = 'Early_Blight'
    elif has_all(text, ['late', 'blight']):
        dis = 'Late_Blight'
    elif has_all(text, ['mold']) or has_all(text, ['mould']):
        dis = 'Leaf_Mold'
    elif has_all(text, ['septoria']):
        dis = 'Septoria'
    elif has_all(text, ['spider']):
        dis = 'Spider_Mite'
    elif has_all(text, ['target']):
        dis = 'Target_Spot'
    elif has_all(text, ['yellow', 'curl']):
        dis = 'Yellow_Leaf_Curl'
    elif has_all(text, ['curl']):
        dis = 'Leaf_Curl'
    elif has_all(text, ['mosaic']):
        dis = 'Mosaic'
    elif has_all(text, ['fusarium']) or has_all(text, ['fussarium']):  # Fusarium only - Verticillium wilt is a different disease, so it is skipped. 'fussarium' is the misspelling the Kaggle cotton folder ships with.
        dis = 'Fusarium_Wilt'
    elif has_all(text, ['healthy']) or has_all(text, ['fresh']):
        dis = 'Healthy'
    elif has_all(text, ['blight']):
        dis = 'Leaf_Blight'
    else:
        return None

    return crop + '_' + dis


# ---- map every image folder to a class ----------------------------------
# DATA_ROOTS is built by Step 2: Kaggle downloads, your own archives, dataset
# folders you dropped in /content, or the tensorflow_datasets fallback.
pool = {}
skipped = []
seen = set()
duplicates = 0
variants = 0

# abdallahalidev/plantvillage-dataset is the spMohanty repo zipped, so every
# class exists three times: color/, grayscale/ and segmented/. The copies keep
# the same filename but differ in file size, so the duplicate filter below
# cannot merge them - and training on grayscale and background-removed copies
# of a leaf already in the pool puts near-identical images on both sides of the
# split. Keep the colour copy only.
VARIANT_DIRS = ('/grayscale/', '/segmented/')

for root in DATA_ROOTS:
    if not os.path.isdir(root):
        continue
    for folder in image_folders(root):
        flat_folder = folder.replace(os.sep, '/').lower()
        if any(marker in flat_folder for marker in VARIANT_DIRS):
            variants += 1
            continue
        label = classify(norm(folder))
        if label is None:
            skipped.append(folder)
            continue
        for filename in sorted(os.listdir(folder)):
            if not filename.lower().endswith(IMG_EXTS):
                continue
            full = os.path.join(folder, filename)
            try:
                # The same photo arriving twice (an archive plus an already
                # extracted copy of it) would put one image on both sides of the
                # train/val split, so only the first copy is kept.
                key = (label, filename.lower(), os.path.getsize(full))
            except OSError:
                continue
            if key in seen:
                duplicates += 1
                continue
            seen.add(key)
            pool.setdefault(label, []).append(full)

assert pool, 'Nothing was mapped to a class - read the Step 2 inventory output.'

# ---- keep only classes the app can actually give advice for -------------
# A class the remedy dictionary cannot match would show the farmer the wrong
# treatment, which is worse than not predicting it at all.
unusable = []
if DROP_CLASSES_WITHOUT_REMEDY:
    for label in sorted(pool):
        rec, score = app_lookup(label)
        if rec is not None and score >= 1 and remedy_quality(label, rec):
            continue
        unusable.append((label, rec.get('disease_name') if rec else None))
    for label, _ in unusable:
        pool.pop(label, None)

assert pool, ('Every class was dropped - the remedy dictionary could not match any class. Set DROP_CLASSES_WITHOUT_REMEDY = False in Step 0.')

# ---- copy into /content/odisha_crops, capped per class ------------------
rng = random.Random(SEED)
total = 0
small = []
print('Class mapping (' + str(len(pool)) + ' classes):')
print('  ' + '-' * 46)
for label in sorted(pool):
    files = sorted(pool[label])
    rng.shuffle(files)
    files = files[:CAP_PER_CLASS]
    dst = os.path.join(odisha_dir, label)
    os.makedirs(dst, exist_ok=True)
    for i, src in enumerate(files):
        shutil.copy2(src, os.path.join(dst, label + '_' + str(i).zfill(5) + '.jpg'))
    total += len(files)
    if len(files) < 100:
        small.append(label)
    print('  ' + label.ljust(26) + str(len(files)).rjust(6) + ' images')
print('  ' + '-' * 46)
print('  ' + str(total) + ' images in ' + str(len(pool)) + ' classes')
if duplicates:
    print('  skipped ' + str(duplicates) + ' duplicate file(s)')
if variants:
    print('  skipped ' + str(variants) + ' grayscale/segmented variant folder(s)')

print()
print('Skipped ' + str(len(skipped)) + ' folder(s) that matched no crop+disease rule.')
for s in skipped[:12]:
    print('  ' + s)
if len(skipped) > 12:
    print('  ... and ' + str(len(skipped) - 12) + ' more')

cotton_skipped = [s for s in skipped if 'cotton' in s.lower()]
if cotton_skipped:
    print()
    print('WARNING - ' + str(len(cotton_skipped)) + ' cotton folder(s) were skipped.')
    print('If those folders are named generically (diseased / fresh) there is no')
    print('way to turn them into a disease class. Check the paths printed above,')
    print('then either label them by hand or drop cotton from KAGGLE_DATASETS.')

if unusable:
    print()
    print('Dropped ' + str(len(unusable)) + ' class(es) the app has no remedy for:')
    for label, matched in unusable:
        print('  ' + label.ljust(26) + '-> ' + str(matched))
    print('These images are still on disk - just not trained on. To use them, set')
    print('DROP_CLASSES_WITHOUT_REMEDY = False in Step 0 and add a record with')
    print('matching keywords to src/data/offline_diseases.json.')

if small:
    print()
    print('WARNING - these classes have fewer than 100 images, expect weak results:')
    print('  ' + ', '.join(small))


---
## Step 3b - Class contract + the `Other_NotALeaf` class

Two jobs, both about the app rather than the maths.

**1. The crop prefix is load-bearing.** `src/services/offlineDiagnosis.js` masks
predictions by the part before the underscore (`Paddy_*`, `Tomato_*`, ...), so a
class named anything else can never be selected for a crop. This cell prints any
class that no crop prefix covers - the failure that would otherwise show up on the
phone as "this disease is never predicted".

**2. Add one class that means "not a leaf of the crop you picked".** Without it, a
photo of soil, a hand or another plant still gets a confident disease name. The app
checks this class on the **raw** probabilities *before* the crop mask
(`GLOBAL_PREFIXES` / `OTHER_MIN_CONFIDENCE = 0.6`), so the prefix must be `Other_`.

Real negatives from `NEGATIVE_DIRS` are used first. If none are found, the cell
synthesises placeholders from the dataset itself (blown out, black, destroyed
detail, pure noise) and **prints a warning** - those teach the model "unusable
photo", not "soil". Upload 150-300 real negatives before you present.


In [ ]:
# ============================================================
# STEP 3b - Class contract check + the Other_NotALeaf class
# ============================================================

# Must match CROP_PREFIXES in src/services/offlineDiagnosis.js
APP_CROP_PREFIXES = ['Paddy', 'Cotton', 'Tomato', 'Potato', 'Maize']

existing = sorted(d for d in os.listdir(odisha_dir)
                  if os.path.isdir(os.path.join(odisha_dir, d)))
print('Classes built in Step 3: ' + str(len(existing)))

unprefixed = [n for n in existing if n.split('_')[0] not in APP_CROP_PREFIXES]
if unprefixed:
    print()
    print('WARNING - the app can never select these classes, because their crop')
    print('prefix is not in CROP_PREFIXES ' + str(APP_CROP_PREFIXES) + ':')
    for name in unprefixed:
        print('  ' + name)
    print('Either rename them to <Crop>_<Disease> or drop them.')
else:
    print('OK: every class starts with a crop prefix the app recognises.')

# ---- optional: is there anything to merge? --------------------------------
# Two diseases that carry the SAME remedy cannot change what the farmer does, so
# merging them would be free accuracy. As of today the dictionary has no such
# pair: 30 records -> 30 distinct treatments. Re-run this when it changes.
# (Cross-crop merges are impossible: the crop mask needs the prefix.)
if DICTIONARY_PATH and os.path.exists(DICTIONARY_PATH):
    _rows = json.load(open(DICTIONARY_PATH, encoding='utf-8'))
    _groups = {}
    for _r in _rows:
        _key = (_r['crop_name'],
                _r['organic_remedy'].strip().lower(),
                _r['chemical_remedy'].strip().lower())
        _groups.setdefault(_key, []).append(_r['id'])
    _dupes = [v for v in _groups.values() if len(v) > 1]
    print()
    print('Dictionary: ' + str(len(_rows)) + ' records -> ' + str(len(_groups))
          + ' distinct within-crop remedy branches')
    if _dupes:
        for _ids in _dupes:
            print('  could be ONE class: ' + str(_ids))
    else:
        print('  nothing to merge - every record needs its own class today')
else:
    print()
    print('Set DICTIONARY_PATH in Step 0 to the app copy of offline_diseases.json')
    print('to check whether any two records share a treatment (mergeable class).')

# ---- the Other class ------------------------------------------------------
other_dir = os.path.join(odisha_dir, OTHER_CLASS)

if not TRAIN_OTHER_CLASS:
    print()
    print('TRAIN_OTHER_CLASS is False - no ' + OTHER_CLASS + ' class will be trained.')
    print('The app still has the pixel guard, but a photo of soil will keep')
    print('getting a disease name.')
else:
    shutil.rmtree(other_dir, ignore_errors=True)
    os.makedirs(other_dir, exist_ok=True)

    real_negatives = []
    for src_dir in NEGATIVE_DIRS:
        if os.path.isdir(src_dir):
            for root, _dirs, files in os.walk(src_dir):
                for fname in files:
                    if fname.lower().endswith(IMG_EXTS):
                        real_negatives.append(os.path.join(root, fname))
    real_negatives = real_negatives[:NEGATIVE_LIMIT]

    for i, path in enumerate(real_negatives):
        ext = os.path.splitext(path)[1].lower()
        shutil.copy2(path, os.path.join(other_dir, 'neg_' + str(i).zfill(4) + ext))

    print()
    if real_negatives:
        print('Other: ' + str(len(real_negatives)) + ' REAL negatives from ' + str(NEGATIVE_DIRS))
    else:
        print('Other: no real negatives found in ' + str(NEGATIVE_DIRS))

    # Placeholders so the class is never empty. Real photos are what teach this
    # class - see MODEL_PLAN.md section 3.
    need = max(0, NEGATIVE_LIMIT - len(real_negatives))
    if need:
        print('Other: synthesising ' + str(need) + ' PLACEHOLDER negatives')
        print('       (blown out / black / detail destroyed / pure noise)')
        print('       THESE ARE NOT FIELD PHOTOS - upload real ones before you present.')

        rng = np.random.default_rng(SEED)
        pool = []
        for name in existing:
            class_dir = os.path.join(odisha_dir, name)
            files = sorted(f for f in os.listdir(class_dir)
                           if f.lower().endswith(IMG_EXTS))
            if files:
                picks = rng.choice(files, size=min(4, len(files)), replace=False)
                pool.extend(os.path.join(class_dir, str(f)) for f in picks)
        if not pool:
            raise RuntimeError('No images available to synthesise negatives from.')

        for i in range(need):
            raw = tf.io.read_file(pool[i % len(pool)])
            img = tf.io.decode_image(raw, channels=3, expand_animations=False)
            img = tf.image.resize(img, IMG_SIZE)
            img = tf.cast(img, tf.float32) / 255.0

            kind = i % 4
            if kind == 0:
                img = tf.clip_by_value(tf.image.adjust_brightness(img, 0.45), 0.0, 1.0)
            elif kind == 1:
                img = tf.clip_by_value(tf.image.adjust_brightness(img, -0.45), 0.0, 1.0)
            elif kind == 2:
                img = tf.image.resize(tf.image.resize(img, (28, 28)), IMG_SIZE)
            else:
                img = tf.clip_by_value(
                    tf.random.normal(tf.shape(img), mean=0.5, stddev=0.25, seed=i),
                    0.0, 1.0)

            out = os.path.join(other_dir, 'syn_' + str(i).zfill(4) + '.png')
            tf.keras.utils.save_img(out, tf.cast(img * 255.0, tf.uint8))
            if (i + 1) % 100 == 0:
                print('   ' + str(i + 1) + ' / ' + str(need))

    total_other = len([f for f in os.listdir(other_dir) if f.lower().endswith(IMG_EXTS)])
    print('Other: ' + str(total_other) + ' images in the ' + OTHER_CLASS + ' class')

# ---- final class list -----------------------------------------------------
FINAL_CLASSES = sorted(d for d in os.listdir(odisha_dir)
                       if os.path.isdir(os.path.join(odisha_dir, d)))
print()
print('Final class list (' + str(len(FINAL_CLASSES)) + ') - this is the order')
print('Step 4 reads and classes.json will record:')
for i, name in enumerate(FINAL_CLASSES):
    print('  ' + str(i).rjust(3) + '  ' + name)

_counts = {name: len([f for f in os.listdir(os.path.join(odisha_dir, name))
                      if f.lower().endswith(IMG_EXTS)])
           for name in FINAL_CLASSES}
_thin = {n: c for n, c in _counts.items() if c < 50}
if _thin:
    print()
    print('Thin classes (<50 images) - expect poor recall from these:')
    for name, count in sorted(_thin.items(), key=lambda kv: kv[1]):
        print('  ' + name.ljust(30) + str(count).rjust(5))


---
## Step 4 - Load, split and augment

`class_names` is passed in explicitly and sorted, so the index of every class is
identical to the order later written into `classes.json`. Augmentation lives in the
dataset pipeline and **never inside the model**: anything inside the model is exported
into `model.json` and would then run on every inference on the phone.

Order matters: augmentation runs on the raw 0-255 images and the divide by 255 happens
last, because `RandomBrightness` and `RandomContrast` add an absolute offset in the range
the layer expects. Validation is never augmented and stays exactly 0-1.

Step 4 asserts both halves of that assumption - the layer's `value_range` really is
`(0, 255)` and the images feeding it really do peak above 1.5 - because a `rescale=` in
the dataset, or dividing before augmenting, would silently turn every augmented image
into a flat white or black rectangle.

No `.cache()` here - ~20k images at 224x224 float32 does not fit in Colab RAM.


In [ ]:
# ============================================================
# STEP 4 - Build the tf.data pipelines
# ============================================================
CLASS_NAMES = sorted([d for d in os.listdir(odisha_dir)
                      if os.path.isdir(os.path.join(odisha_dir, d))])
NUM_CLASSES = len(CLASS_NAMES)
assert NUM_CLASSES >= 2, 'Need at least 2 classes - see the Step 3 output.'

AUTOTUNE = tf.data.AUTOTUNE
print('Loading ' + str(NUM_CLASSES) + ' classes from ' + odisha_dir)

train_ds = tf.keras.utils.image_dataset_from_directory(
    odisha_dir,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset='training',
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    odisha_dir,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset='validation',
)

# Field-style augmentation. Note this pipeline sits OUTSIDE the model (it runs in
# prep_train below), so none of it is exported and the app's contract is
# untouched - the phone still feeds plain 0-1 pixels.
if FIELD_AUGMENT:
    augment = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(AUG_ROTATION),
        tf.keras.layers.RandomZoom(AUG_ZOOM),
        tf.keras.layers.RandomTranslation(AUG_TRANSLATION, AUG_TRANSLATION),
        tf.keras.layers.RandomBrightness(AUG_BRIGHTNESS),
        tf.keras.layers.RandomContrast(AUG_CONTRAST),
        tf.keras.layers.Lambda(lambda t: t + tf.random.normal(tf.shape(t), stddev=AUG_NOISE), name='sensor_noise'),
    ], name='augmentation')
else:
    # The previous, gentler pipeline - kept so you can compare the two.
    augment = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.2),
        tf.keras.layers.RandomBrightness(0.1),
        tf.keras.layers.RandomContrast(0.1),
    ], name='augmentation')


# The augmentation layers above are calibrated for 0-255 pixels: RandomBrightness
# adds factor * value_range[1] as an ABSOLUTE offset, so 0.25 means +63.75 on
# [0,255]. prep_train feeds it raw 0-255 images and divides by 255 afterwards.
# If that ever changed - a rescale= in the dataset, or dividing before augmenting
# - the same +63.75 would land on pixels already in [0,1] and every augmented
# image would clip to pure white or black, with nothing raising anywhere.
_bright = [l for l in augment.layers if isinstance(l, tf.keras.layers.RandomBrightness)]
_vr = tuple(_bright[0].value_range) if _bright else None
assert _vr in (None, (0, 255)), (
    'RandomBrightness value_range is ' + str(_vr) + ' but prep_train feeds it '
    '0-255 pixels - the brightness offset would clip every image.')

_raw_batch = next(iter(train_ds.take(1)))[0]
_raw_min, _raw_max = float(_raw_batch.numpy().min()), float(_raw_batch.numpy().max())
assert _raw_max > 1.5, (
    'raw images peak at ' + str(_raw_max) + ' - they are already in [0,1], so the '
    'augmentation would destroy them. Do not pass rescale= to '
    'image_dataset_from_directory, and do not divide by 255 before augmenting.')
print('Augmentation contract: value_range ' + str(_vr)
      + ', raw pixels ' + str(round(_raw_min, 1)) + ' .. ' + str(round(_raw_max, 1)))

def prep_train(x, y):
    # Augment FIRST, in the native 0-255 range: RandomBrightness adds an absolute
    # offset, so handing it [0,1] images would turn the picture almost black.
    x = augment(x, training=True)
    x = x / 255.0                     # [0,1] - exactly what the app feeds the model
    return x, y


def prep_eval(x, y):
    return x / 255.0, y               # validation is never augmented


train_ds = train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

xt, yt = next(iter(train_ds))
xv, yv = next(iter(val_ds))
print()
print('Train batch shape : ' + str(xt.shape))
print('Train pixel range : ' + str(round(float(xt.numpy().min()), 3)) + ' .. '
      + str(round(float(xt.numpy().max()), 3))
      + '   (0..1, plus a little augmentation overshoot)')
print('Val pixel range   : ' + str(round(float(xv.numpy().min()), 3)) + ' .. '
      + str(round(float(xv.numpy().max()), 3)) + '   (must be inside 0.0 .. 1.0)')
assert float(xv.numpy().min()) >= 0.0, 'validation pixels below 0 - contract broken'
assert float(xv.numpy().max()) <= 1.0, 'validation pixels above 1 - contract broken'

print()
print('Class index order (this becomes classes.json):')
for i, name in enumerate(CLASS_NAMES):
    print('  ' + str(i).rjust(3) + '  ' + name)


---
## Step 5 - Build the model

MobileNetV2 (ImageNet weights) with a small classification head. The first layer
rescales the app's `[0, 1]` pixels to the `[-1, 1]` range that MobileNetV2 expects, and
it is deliberately part of the model so the phone does not have to care.


In [ ]:
# ============================================================
# STEP 5 - Build the model
# ============================================================
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights='imagenet',
    alpha=ALPHA,
)
base_model.trainable = False

model = tf.keras.Sequential([
    # [0,1] (what the app sends) -> [-1,1] (what MobileNetV2 wants).
    # Baked into model.json on export - do NOT remove it.
    tf.keras.layers.Rescaling(scale=2.0, offset=-1.0, name='input_rescale'),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', name='predictions'),
], name='krishisetu_crop_doctor')

# Keras 3 (what Colab ships) only auto-builds a Sequential when its FIRST layer
# is an Input layer or already exposes an input_shape. Ours starts with
# Rescaling, which has neither, so the model stays unbuilt and every one of
# model.count_params(), model.input_shape and model.output_shape raises
# "the layer isn't built". Building it explicitly fixes that.
# Note the shape INCLUDES the batch dimension - that is what build() expects.
if not model.built:
    model.build((None, IMG_SIZE[0], IMG_SIZE[1], 3))

assert model.built, 'the model did not build - check the layer stack above'
assert model.layers[0].__class__.__name__ == 'Rescaling', (
    'first layer must be Rescaling - the app sends [0,1] and MobileNetV2 wants [-1,1]')
assert model.layers[0].name == 'input_rescale', 'the first layer must be input_rescale'

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()
print()
print('Parameters  : ' + str(model.count_params()))
print('Input shape : ' + str(model.input_shape))
print('Output shape: ' + str(model.output_shape))


---
## Step 6 - Train, phase 1 (MobileNetV2 frozen)

The callbacks are built fresh for each phase on purpose: `EarlyStopping` and
`ReduceLROnPlateau` remember their best value, and reusing the same instances across two
`fit` calls would make phase 2 stop immediately.

If phase 1 stops early that is normal and good - it means validation accuracy stopped
improving.


In [ ]:
# ============================================================
# STEP 6 - Phase 1: train the classifier head
# ============================================================
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=5,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=2, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            '/content/best_model.keras', monitor='val_accuracy',
            save_best_only=True, verbose=1),
    ]


print('PHASE 1 - classifier head, MobileNetV2 frozen')
history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P1,
                     callbacks=make_callbacks(), verbose=1)
print()
print('Phase 1 best val_accuracy: '
      + str(round(float(max(history1.history['val_accuracy'])), 4)))


---
## Step 7 - Train, phase 2 (fine-tune)

The last 30 layers of MobileNetV2 are unfrozen and trained at a 10x lower learning rate.
If validation accuracy gets *worse* than the end of phase 1, lower the learning rate to
`0.00005` or unfreeze only the last 15 layers.


In [ ]:
# ============================================================
# STEP 7 - Phase 2: fine-tune the top of MobileNetV2
# ============================================================
print('PHASE 2 - fine-tuning the last 30 layers of MobileNetV2')

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P2,
                     callbacks=make_callbacks(), verbose=1)
print()
print('Phase 2 best val_accuracy: '
      + str(round(float(max(history2.history['val_accuracy'])), 4)))


---
## Step 8 - Evaluate

Accuracy/loss curves, a per-class classification report, and a confusion matrix.
Look for classes that are flat at 0.00 - that is a sign of too few images or of a
class name that never appeared in the source data.


In [ ]:
# ============================================================
# STEP 8a - Training curves
# ============================================================
acc = history1.history['accuracy'] + history2.history['accuracy']
vacc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
vloss = history1.history['val_loss'] + history2.history['val_loss']
epochs = range(1, len(acc) + 1)
phase_split = len(history1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc, 'b-o', label='train', markersize=4)
ax1.plot(epochs, vacc, 'r-o', label='val', markersize=4)
ax1.axvline(x=phase_split, color='gray', linestyle='--', alpha=0.6)
ax1.set_title('Accuracy')
ax1.set_xlabel('epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, loss, 'b-o', label='train', markersize=4)
ax2.plot(epochs, vloss, 'r-o', label='val', markersize=4)
ax2.axvline(x=phase_split, color='gray', linestyle='--', alpha=0.6)
ax2.set_title('Loss')
ax2.set_xlabel('epoch')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# STEP 8b - Classification report + confusion matrix
# ============================================================
y_true = []
y_pred = []

for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

overall = sum(1 for t, p in zip(y_true, y_pred) if t == p) / max(1, len(y_true))
print('Validation accuracy: ' + str(round(overall * 100, 2)) + '%'
      + '  (' + str(len(y_true)) + ' images)')
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                            digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(max(10, NUM_CLASSES * 0.7), max(8, NUM_CLASSES * 0.5)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Confusion matrix - validation set')
ax.set_xlabel('predicted')
ax.set_ylabel('actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


---
## Step 9 - Verify against the app contract

This is the cell that prevents the classic silent failure: a model that exports fine but
scores 30% on the phone because the preprocessing or the class order does not match.

It checks the input/output shapes and the `Rescaling` layer, then replays the app's own
fuzzy remedy matcher over every class you trained and reports any class that has no remedy
record. Finally it **asserts** that every crop in the app's picker (`Paddy`, `Maize`,
`Tomato`, `Potato`, `Cotton`) has at least one trained class, so a missing crop fails the
notebook instead of quietly reaching the phone as "Crop Not Supported". The dictionary is embedded in Step 3, so **there is nothing to upload**; dropping
the real `src/data/offline_diseases.json` at `/content/offline_diseases.json` only makes
the check run against the live file instead of the embedded copy.


In [ ]:
# ============================================================
# STEP 9 - Contract checks
# ============================================================
print('Input shape  : ' + str(model.input_shape))
print('Output shape : ' + str(model.output_shape))

assert model.input_shape[1:] == (IMG_SIZE[0], IMG_SIZE[1], 3), 'wrong input shape'
assert model.output_shape[-1] == NUM_CLASSES, 'output classes != NUM_CLASSES'

first_layer = model.layers[0]
cfg = first_layer.get_config()
assert first_layer.__class__.__name__ == 'Rescaling', 'first layer must be Rescaling'
assert float(cfg['scale']) == 2.0, 'Rescaling scale must be 2.0'
assert float(cfg['offset']) == -1.0, 'Rescaling offset must be -1.0'

print()
print('OK - the app sends [0,1] as 224x224x3, and this model:')
print('  layer 0  Rescaling(scale=2.0, offset=-1.0)  ->  [-1,1] for MobileNetV2')
print('  output   ' + str(NUM_CLASSES) + ' classes, index order becomes classes.json')

# ---- replay the app's remedy matcher ------------------------------------
# app_normalize / app_lookup / remedy_quality / REMEDY_DB all come from Step 3,
# so there is nothing to upload. If you restart the runtime, Step 3 has to run
# again before this cell - the assert below catches that.
DB_PATH = '/content/offline_diseases.json'
if os.path.exists(DB_PATH):
    with open(DB_PATH) as f:
        REMEDY_DB = json.load(f)
    print('Using the uploaded ' + DB_PATH + ' instead of the embedded copy.')

assert 'app_lookup' in dir(), 'Run Step 3 first - it defines app_lookup().'

print()
print('Remedy coverage vs the app dictionary (' + str(len(REMEDY_DB)) + ' records):')
missing = []
for name in CLASS_NAMES:
    rec, score = app_lookup(name)
    if rec is None or score < 1:
        missing.append(name)
    elif remedy_quality(name, rec):
        print('  ok   ' + name.ljust(26) + '-> ' + str(rec.get('disease_name')))
    else:
        print('  WARN ' + name.ljust(26) + '-> ' + str(rec.get('disease_name')))
        print('       ^ these share only generic words, so the app would give the')
        print('         WRONG remedy. Step 3 drops these classes when')
        print('         DROP_CLASSES_WITHOUT_REMEDY = True.')

if missing:
    print()
    print('NO REMEDY FOUND for: ' + ', '.join(missing))
    print('Step 3 drops these classes when DROP_CLASSES_WITHOUT_REMEDY = True,')
    print('so you should not see this line. To keep one, add a record with a')
    print('matching crop_name and disease keywords to src/data/offline_diseases.json.')
else:
    print()
    print('Every trained class maps to a remedy record.')

# ---- crop coverage: every crop the picker offers needs a trained class ----
# The crop picker is not built from the model, so a crop with zero trained
# classes stays selectable. src/services/offlineDiagnosis.js treats that as its
# own outcome ('Crop Not Supported') instead of masking the crop away and
# naming some other crop's disease, so it is safe - but the farmer still gets
# no offline answer for that crop.
APP_CROPS = ['Paddy', 'Maize', 'Tomato', 'Potato', 'Cotton']
by_crop = {}
for name in CLASS_NAMES:
    by_crop.setdefault(str(name).split('_')[0], []).append(name)

print()
print('Crop coverage (the picker offers ' + str(len(APP_CROPS)) + ' crops):')
uncovered = [c for c in APP_CROPS if c not in by_crop]
for crop in APP_CROPS:
    if crop in by_crop:
        print('  ok   ' + crop.ljust(8) + str(len(by_crop[crop])).rjust(3) + ' class(es)')
    else:
        print('  MISS ' + crop.ljust(8) + '  0 classes')

if uncovered:
    print()
    print('No trained class for: ' + ', '.join(uncovered))
    print('The app would answer "Crop Not Supported" for those crops - safe, but')
    print('not an answer to a farmer holding a leaf. Add a dataset that contains')
    print('them to KAGGLE_DATASETS in Step 0 and re-run from Step 2:')
    print('  Maize    <- abdallahalidev/plantvillage-dataset (corn folders)')
    print('  Paddy    <- anshulm257/rice-disease-dataset')
    print('  Cotton   <- seroshkarim/cotton-leaf-disease-dataset')
    print()
    print('(Training with a crop deliberately missing? Delete the assert below -')
    print('  the app treats it as its own outcome rather than guessing, but this')
    print('  notebook stops so that choice has to be made on purpose.)')
    print()
assert not uncovered, ('the crop picker offers crops this model cannot answer for: '
                       + ', '.join(uncovered) + '. See the Step 2 output.')
print('Every crop in the picker has at least one trained class.')


---
## Step 10 - Export to TensorFlow.js Layers format

The app loads the model with `tf.loadLayersModel('/model/model.json')`, so the export
must be **Layers** format. Do not switch to a TF SavedModel: that produces a Graph model
(`loadGraphModel`), which this app cannot load.

`float16` quantization roughly halves the weight file and is safe here - TF.js
dequantizes to float32 when it loads. If it errors for any reason, the cell falls back to
float32 automatically.


In [ ]:
# ============================================================
# STEP 10 - Export model.json + weight shards + classes.json
# ============================================================
TFJS_DIR = '/content/tfjs_model'
shutil.rmtree(TFJS_DIR, ignore_errors=True)
os.makedirs(TFJS_DIR, exist_ok=True)

quant = {'float16': '*'} if EXPORT_FLOAT16 else None


def export_with_save_keras_model(out_dir, quantization):
    # The normal path, and the one to try first.
    tfjs.converters.save_keras_model(model, out_dir, quantization_dtype_map=quantization)


def export_with_keras3_reader(out_dir, quantization):
    # Fallback. save_keras_model() only reads Keras-2 HDF5, even though the same
    # package ships a Keras-3 reader it never calls. Save the native .keras
    # archive and hand its three parts to that reader ourselves.
    import h5py
    from tensorflowjs.converters.keras_h5_conversion import (
        h5_v3_merged_saved_model_to_tfjs_format, write_artifacts)
    archive, work = '/content/_export.keras', '/content/_export_keras3'
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(work, exist_ok=True)
    model.save(archive)
    with zipfile.ZipFile(archive) as zf:
        config_file = json.loads(zf.read('config.json'))
        meta_file = json.loads(zf.read('metadata.json'))
        zf.extract('model.weights.h5', work)
    with h5py.File(os.path.join(work, 'model.weights.h5'), 'r') as h5:
        topology, groups = h5_v3_merged_saved_model_to_tfjs_format(
            h5, meta_file, config_file)
    write_artifacts(topology, groups, out_dir,
                    quantization_dtype_map=quantization,
                    metadata={'generatedBy': 'krishisetu notebook (keras 3 reader)'})
    shutil.rmtree(work, ignore_errors=True)
    if os.path.exists(archive):
        os.remove(archive)


ATTEMPTS = [
    ('save_keras_model, float16', export_with_save_keras_model, quant),
    ('save_keras_model, float32', export_with_save_keras_model, None),
    ('keras-3 reader, float16', export_with_keras3_reader, quant),
    ('keras-3 reader, float32', export_with_keras3_reader, None),
]

QUANTIZATION = None
for _label, _export, _quant in ATTEMPTS:
    shutil.rmtree(TFJS_DIR, ignore_errors=True)
    os.makedirs(TFJS_DIR, exist_ok=True)
    try:
        _export(TFJS_DIR, _quant)
        QUANTIZATION = 'float16' if _quant else 'float32'
        print('Exported to TF.js Layers format via ' + _label)
        break
    except Exception as _exc:
        print('  ' + _label + ' failed: ' + repr(_exc))
else:
    raise RuntimeError('Every TensorFlow.js export path failed. Read the four errors above.')

# ---- strip Keras-3-only keys from the exported topology ------------------
# modelTopology is copied straight from the config the exporter read, and a Keras 3
# config carries "module" / "registered_name" / "date_saved" keys that the
# TensorFlow.js loader does not expect. Drop them recursively so the phone gets a
# clean model.json whichever export path produced it.

def strip_keras3_keys(obj, removed):
    if isinstance(obj, dict):
        for key in list(obj.keys()):
            if key in ('module', 'registered_name', 'date_saved'):
                del obj[key]
                removed.append(key)
            else:
                strip_keras3_keys(obj[key], removed)
    elif isinstance(obj, list):
        for item in obj:
            strip_keras3_keys(item, removed)


MANIFEST_PATH = os.path.join(TFJS_DIR, 'model.json')
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
_stripped = []
strip_keras3_keys(manifest, _stripped)
if _stripped:
    with open(MANIFEST_PATH, 'w') as f:
        json.dump(manifest, f)
print('Stripped ' + str(len(_stripped)) + ' Keras-3-only key(s) from model.json')

# classes.json - a plain JSON array, in the model's output order
with open(os.path.join(TFJS_DIR, 'classes.json'), 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

# metadata.json - optional, the app ignores it, useful for your own records
metadata = {
    'model_name': 'KrishiSetu Odisha Crop Disease Classifier',
    'architecture': 'MobileNetV2 alpha=' + str(ALPHA) + ' + transfer learning',
    'input_size': [224, 224, 3],
    'input_range': [0.0, 1.0],
    'first_layer': 'Rescaling(scale=2.0, offset=-1.0)',
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'quantization': QUANTIZATION,
    'val_accuracy': round(float(max(history2.history['val_accuracy'])), 4),
    'parameters': int(model.count_params()),
    'trained_on': 'Google Colab, T4 GPU',
}
with open(os.path.join(TFJS_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

# ---- validate the export before trusting it ------------------------------
manifest = json.load(open(os.path.join(TFJS_DIR, 'model.json')))
assert manifest.get('format') == 'layers-model', (
    'Export is not a Layers model (got ' + str(manifest.get('format')) + ').')
topology = manifest.get('modelTopology') or {}
assert topology.get('model_config'), (
    'model.json has no modelTopology - the app would fail to load it.')
groups = manifest.get('weightsManifest') or []
assert groups, 'model.json has no weightsManifest.'
shard_paths = [p for g in groups for p in g.get('paths', [])]
for rel in shard_paths:
    full = os.path.join(TFJS_DIR, rel)
    assert os.path.exists(full), 'missing weight shard ' + rel
    assert os.path.getsize(full) > 0, 'empty weight shard ' + rel

exported_layers = topology['model_config'].get('config', {}).get('layers', [])
print()
print('Exported layers     : ' + str(len(exported_layers)))
print('First exported layer: '
      + str(exported_layers[0].get('class_name') if exported_layers else '?'))
print('Weight shards       : ' + str(len(shard_paths)))
print('classes.json entries: ' + str(NUM_CLASSES))
print()

running_total = 0
print('Files in ' + TFJS_DIR + ':')
for filename in sorted(os.listdir(TFJS_DIR)):
    size = os.path.getsize(os.path.join(TFJS_DIR, filename))
    running_total += size
    print('  ' + filename.ljust(28) + str(round(size / 1000000, 2)).rjust(8) + ' MB')
print('  ' + '-' * 36)
print('  ' + 'TOTAL'.ljust(28) + str(round(running_total / 1000000, 2)).rjust(8) + ' MB')
print()
if running_total > 15000000:
    print('Tip: for a smaller download set ALPHA = 0.5 in Step 0 (about 4x smaller).')


---
## Step 11 - Package and download

Creates `/content/tfjs_model.zip` and downloads it to your computer.


In [ ]:
# ============================================================
# STEP 11 - Zip it and download it
# ============================================================
ZIP_PATH = '/content/tfjs_model.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive('/content/tfjs_model', 'zip', TFJS_DIR)
print('Created ' + ZIP_PATH + ' ('
      + str(round(os.path.getsize(ZIP_PATH) / 1000000, 2)) + ' MB)')

if BACKUP_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        shutil.copy2(ZIP_PATH, '/content/drive/MyDrive/tfjs_model.zip')
        print('Backed up to /content/drive/MyDrive/tfjs_model.zip')
    except Exception as exc:
        print('Drive backup skipped (' + str(exc) + ')')

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as exc:
    print('Auto-download unavailable (' + str(exc) + ')')
    print('Use the Files panel: right-click tfjs_model.zip > Download')

print()
print('=' * 60)
print('INSTALL INTO THE APP - run this on your machine, in KrishiSetu-AI/')
print('=' * 60)
print('  mkdir -p public/model')
print('  unzip -o tfjs_model.zip -d public/model')
print('  ls public/model   # model.json, group1-shard*.bin, classes.json')
print('  npm run dev')
print()
print('Then test the offline path: DevTools > Network > Offline (or airplane mode)')
print('and scan a leaf photo. Online the app prefers Gemini; the local model is')
print('only used when the device is offline.')


---
## Step 12 - Troubleshooting and useful facts

### Step 2 says NO DATASET FOUND
Three ways, pick one:

* **Kaggle account (best)** - create an access token at `kaggle.com > your avatar >
Settings > API > Create New Token`, paste it into `KAGGLE_API_TOKEN` in Step 0 and re-run
Step 2. This is the only path that gives you Paddy and Cotton as well as PlantVillage.
  * A `KGAT_` token is sent as an HTTP `Bearer` header. It is **not** a `kaggle.json`
  username/key pair, so pasting it into a `kaggle.json` can never work. If you have an
  old-style username + key instead, put them in `KAGGLE_USERNAME` / `KAGGLE_KEY`.
  * Step 2 prints `Kaggle auth: access token KGAT_xxxx...xxxx` when it accepted the
  credential. If it prints `No Kaggle credentials found`, the token never reached it.
  * `401` / `403` from both the CLI and the REST fallback means the token is wrong or
  expired, or you have not accepted that dataset's rules once in a browser.
* **Upload the zips yourself** - set `UPLOAD_NOW = True` in Step 0 and re-run Step 2; a file
dialog opens. Any `.zip`/`.tar.gz` in `/content`, or a dataset folder you already
extracted into `/content`, is picked up automatically.
* **No account at all** - `USE_TFDS_FALLBACK = True` (the default) pulls PlantVillage
through `tensorflow_datasets` for any picker crop that still has no folder on disk. It
supplies maize/tomato/potato and never Paddy or Cotton, so it complements the Kaggle
downloads instead of replacing them. Step 2 prints crop coverage before and after, so you
can see exactly what is still missing.

### Your token is in a git-tracked file
The default here is `KAGGLE_API_TOKEN = ''`. **If you ever pasted one, rotate it now**
(`kaggle.com > Settings > API > Create New Token`): blanking the file does not help,
because git history keeps the old value forever. Read it instead from the Colab secret
`KAGGLE_API_TOKEN`, an env var, or `/content/kaggle_token.txt`.

### pip / import problems in Step 1
Run `Runtime > Restart session`, then run Step 1 again. The cell is defensive about
the known Colab + Python 3.13 + tensorflowjs 4.x issues, but a stale import from an
earlier run is the one thing only a restart clears.

### The app says Model Not Installed
The three files must sit at exactly `public/model/model.json`,
`public/model/group1-shard1of1.bin` and `public/model/classes.json`. They are **tracked
by git**: Vercel builds from a git checkout and this repo has no `.vercelignore`, so if
they were ignored they would never reach the hosted site and Settings > Download would
fail for every user with a 404. Copy them in, commit and push. The service worker
precaches them too, so after replacing a model do a hard reload (or DevTools >
Application > Clear storage) or you keep serving the old weights.

### Offline accuracy is bad
Check, in this order:

1. `classes.json` matches the model's output order (this notebook guarantees it - a
hand-edited file does not).
2. The `Rescaling(scale=2.0, offset=-1.0)` layer is still the first layer (Step 9
asserts it).
3. More images per class: raise `CAP_PER_CLASS`, or drop classes with under 100 images.
4. The leaf fills the frame and is well lit - the model only sees a 224x224 crop.

### Accuracy looks too good
PlantVillage images are studio photos of single leaves on plain backgrounds. Real phone
photos in a paddy field are much harder. Expect a healthy drop in the field, and
re-train with field photos as soon as you have some.

### Model size targets

| Setting | Approx. size |
|---|---|
| `ALPHA = 1.0`, float16 | ~4-6 MB |
| `ALPHA = 1.0`, float32 | ~9-11 MB |
| `ALPHA = 0.5`, float16 | ~1.5-2 MB |

### The class names are the contract
`Paddy_Blast` becomes the prediction string `Paddy_Blast` in the app, which is matched
keyword-wise against `src/data/offline_diseases.json`. Keep the `Crop_Disease` naming and
the five crop prefixes (`Paddy`, `Maize`, `Tomato`, `Potato`, `Cotton`) intact, or the
remedy lookup returns the generic fallback text.

Step 3 and Step 9 both check this for you. `DROP_CLASSES_WITHOUT_REMEDY = True` drops any
class whose only match is generic wording - `Paddy_Leaf_Scald` was dropped for exactly
that reason until `src/data/offline_diseases.json` gained a Leaf Scald record, because
returning bacterial-blight advice for it would be worse than not predicting it - and
Step 9 prints a `WARN` line for anything the dictionary matches only by accident.

### Record your result

```
Model: KrishiSetu Odisha Crop Disease Classifier
Architecture: MobileNetV2 alpha=<ALPHA> + transfer learning
Epochs: <EPOCHS_P1> + <EPOCHS_P2>
Classes: <NUM_CLASSES>
Validation accuracy: ____ %
Weight file: ____ MB (<float16|float32>)
Trained on: Google Colab, T4 GPU
```
